# Getting started — ATT&CK-for-ICS dataset coverage toolkit

This toolkit maps the attacks in **ten public ICS/IIoT intrusion-detection datasets** to **MITRE ATT&CK for ICS (v19.1)** at the *technique* level, with every assignment grounded in a verbatim quote from the dataset's own paper. Companion to the Cyber-AI 2026 paper (0382).

**What you can do with it**
- see which ATT&CK techniques / tactics each dataset exercises;
- find the coverage gaps (techniques/tactics no dataset exercises);
- choose complementary datasets to maximise coverage;
- trace any mapped cell back to the exact source quote;
- reproduce every table and figure, and extend the mapping to new datasets.

**Two counting conventions (important)**
1. **Parent-technique grain** — sub-techniques (e.g. `T1692.002`) are counted under their parent (`T1692`).
2. **Multi-tactic crediting** — a technique is credited to *every* tactic ATT&CK lists it under (so per-tactic counts can exceed the number of distinct techniques).


## 1. Setup

```bash
git clone https://github.com/at-ics-ids/datasets-mapping
cd datasets-mapping
python3 -m venv .venv
.venv/bin/pip install -r requirements.txt      # matplotlib 3.10.9, numpy 2.2.6 (pinned)
.venv/bin/pip install ipykernel                # to run this notebook in VSCode/Jupyter
```
Select the `.venv` kernel, then run the cells below **from the repo root**.

**Layout:** `data/` (the CSVs — the source of truth), `scripts/` (the pipeline), `figures/`, `datacards/` (machine-readable per dataset), `data/EVIDENCE_LOG.md` (all quotes), `reproduce.sh` (one-command rebuild).


In [ ]:
# Load the mapping (no dependencies beyond the standard library)
import os, csv, json, subprocess, itertools
from collections import defaultdict, Counter

ROOT = os.getcwd()
assert os.path.exists(os.path.join(ROOT, "reproduce.sh")), "Run this notebook from the toolkit repo root."
DATA = os.path.join(ROOT, "data")
def load(name): return list(csv.DictReader(open(os.path.join(DATA, name))))
def parent(t): return t.split(".")[0]

mapping = load("technique_mapping_long.csv")
ICS = [r for r in mapping if r["confidence"] in ("high", "medium")]   # counted as ICS coverage
ENT = [r for r in mapping if r["confidence"] == "enterprise"]         # reported separately
datasets = sorted({r["dataset"] for r in mapping})
tname = {r["technique_id"]: r["technique_name"]
         for r in load("attack_ics_v19_1_technique_tactics.csv")}     # id -> display name

print(f"datasets ({len(datasets)}):", datasets)
print("distinct ICS techniques:", len({parent(r['technique_id']) for r in ICS}))
print("ICS assignments        :", len(ICS), " (high/medium)")
print("Enterprise assignments :", len(ENT), " (reported separately)")

## 2. What does a dataset exercise?

In [ ]:
def techniques_of(ds):
    """Distinct parent techniques a dataset exercises, with name / tactic / confidence."""
    out = {}
    for r in ICS:
        if r["dataset"] == ds:
            p = parent(r["technique_id"])
            out[p] = (tname.get(p, ""), r["tactic"], r["confidence"])
    return out

for tid, (name, tac, conf) in techniques_of("SWaT").items():
    print(f"  {tid:8s} {name:34s} [{tac}]  ({conf})")

## 3. Which datasets exercise a given technique?

In [ ]:
def datasets_with(tech):
    tech = parent(tech)
    return sorted({r["dataset"] for r in ICS if parent(r["technique_id"]) == tech})

for t in ["T1692", "T0846", "T0814", "T0847"]:
    ds = datasets_with(t)
    print(f"  {t}  {tname.get(t,''):32s} -> {len(ds)} dataset(s): {ds}")

## 4. Coverage and gaps (Table III)

For each tactic: how many of its techniques any dataset exercises, out of how many ATT&CK-for-ICS v19.1 defines.
The empty tactics are the biggest research gaps.

In [ ]:
TT = defaultdict(set)
for r in load("attack_ics_v19_1_technique_tactics.csv"):
    TT[r["technique_id"]].add(r["tactic"])
available = {r["tactic"]: int(r["techniques_available"]) for r in load("attack_ics_v19_1_catalog.csv")}
TAC = ["Initial Access","Execution","Persistence","Privilege Escalation","Evasion","Discovery",
       "Lateral Movement","Collection","Command and Control","Inhibit Response Function",
       "Impair Process Control","Impact"]
covered = defaultdict(set)
for r in ICS:
    for tac in TT[parent(r["technique_id"])]:
        covered[tac].add(parent(r["technique_id"]))
for tac in TAC:
    bar = "#" * len(covered[tac])
    print(f"  {tac:26s} {len(covered[tac]):>2}/{available.get(tac,'?'):<2} {bar}")
print("\nEmpty tactics (no dataset exercises them):", [t for t in TAC if not covered[t]])

## 5. Per-dataset summary (Table II) — straight from the released file

In [ ]:
for r in load("table2_per_dataset.csv"):
    print(f"  {r['Dataset']:13s} ICS {r['ICS techniques']:>2}  tactics[{r['Tactics covered']}]  ent {r['Enterprise techniques']}")

## 6. Choosing datasets: complementarity & redundancy

Two datasets are **redundant** if they exercise the same techniques, and **complementary** if their sets are disjoint.
The greedy build-up shows the fewest datasets needed to reach maximum coverage.

In [ ]:
sets = {ds: set(techniques_of(ds)) for ds in datasets}

print("Redundant (identical) dataset pairs:")
for a, b in itertools.combinations(datasets, 2):
    if sets[a] and sets[a] == sets[b]:
        print(f"  {a} == {b}   (same {len(sets[a])} techniques)")

best = max(((len(sets[a] | sets[b]), a, b) for a, b in itertools.combinations(datasets, 2)
            if not (sets[a] & sets[b])), default=None)
print("\nLargest complementary (disjoint) pair:", best[1], "+", best[2], f"-> {best[0]} techniques" if best else "")

print("\nGreedy max-coverage build-up:")
cov, remaining = set(), dict(sets)
while remaining:
    ds = max(remaining, key=lambda d: len(sets[d] - cov))
    gain = len(sets[ds] - cov)
    if gain == 0: break
    cov |= sets[ds]; del remaining[ds]
    print(f"  + {ds:13s} (+{gain:>2})  cumulative {len(cov)} / {len({parent(r['technique_id']) for r in ICS})} techniques")

## 7. Trace any cell to its verbatim evidence

In [ ]:
evidence = load("mapping_evidence.csv")
def evidence_for(ds, tech):
    for r in evidence:
        if r["dataset"] == ds and parent(r["technique_id"]) == parent(tech) and r["confidence"] != "removed":
            print(f'{ds} / {r["technique_id"]} {r["technique_name"]}  [{r["confidence"]}]')
            print(f'  quote : "{r["verbatim_evidence"]}"')
            print(f'  source: {r["source"]}')
            if r.get("analyst_note"): print(f'  note  : {r["analyst_note"]}')
            print()

evidence_for("SWaT", "T0832")
evidence_for("ICS-Flow", "T1692.001")

## 8. Enterprise (IT-side) attacks — reported separately, not counted as ICS coverage

In [ ]:
entmap = defaultdict(set)
for r in ENT: entmap[r["dataset"]].add(r["enterprise_ref"])
for ds in datasets:
    if entmap[ds]:
        print(f"  {ds}: {sorted(entmap[ds])}")

# machine-readable per-dataset card:
card = json.load(open(os.path.join(ROOT, "datacards", "SWaT.json")))
print("\nSWaT data card keys:", list(card.keys()))

## 9. Reproduce everything

`reproduce.sh` rebuilds the coverage matrix, Tables II & III, Figures 1–3, the evidence log and the data cards
from `data/technique_mapping_long.csv` — nothing is hand-edited downstream.

In [ ]:
res = subprocess.run(["bash", "reproduce.sh"], cwd=ROOT, capture_output=True, text=True)
print(res.stdout[-600:])
print("exit:", res.returncode)

## 10. Extend the mapping to a new dataset

1. Add rows to **`data/technique_mapping_long.csv`**: `dataset, attack_class, technique_id, technique_name, tactic, confidence, justification, source, enterprise_ref`.
   - `confidence` is `high` (the source names the behaviour), `medium` (it follows by inference), or `enterprise` (IT-side, reported separately).
2. Add the verbatim quote to **`data/mapping_evidence.csv`**; record rejected candidates as `confidence = removed`.
3. If you introduce a technique not yet in **`data/attack_ics_v19_1_technique_tactics.csv`**, add it there with its tactic(s).
4. Run `bash reproduce.sh` — every table, figure, card and the evidence log regenerate.

**Rules the pipeline enforces:** parent-technique grain; a technique credited to every tactic; only `high`/`medium` count as ICS coverage; Enterprise kept separate; nothing counted without a grade.

## 11. Cite
See `CITATION.cff`, or cite the Cyber-AI 2026 paper (0382).


## 12. A quick figure — technique frequency

In [ ]:
import matplotlib.pyplot as plt
freq = Counter()
for ds in datasets:
    for t in set(techniques_of(ds)): freq[t] += 1
items = freq.most_common()
plt.figure(figsize=(11, 3.2))
plt.bar([t for t, _ in items], [c for _, c in items], color="#2E5FA6")
plt.ylabel("number of datasets"); plt.xticks(rotation=90, fontsize=7); plt.tight_layout()
plt.title("How many of the ten datasets exercise each ICS technique")
plt.show()